# 03 · Qwen translation baseline

Baseline: give Qwen a Chinese post, get English back. No slang definitions yet.

Before running (Colab):
1. Change runtime to A100.
2. Upload data/processed/usual_splits.csv (I added this to the code because I kept forgetting to add it)


## 1. Install and import

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
from google.colab import files

uploaded = files.upload() # choose rq1_matched_eval.csv AND rq1_matched_test.csv
print(list(uploaded.keys())) # should print both file names

Saving rq1_matched_eval.csv to rq1_matched_eval.csv
Saving rq1_matched_test.csv to rq1_matched_test.csv
['rq1_matched_eval.csv', 'rq1_matched_test.csv']


In [ ]:
# src for using a HF model (plus Assignment 3): https://medium.com/data-science/choosing-and-implementing-hugging-face-models-026d71426fbe$0
%pip install -U transformers accelerate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.1/13.1 MB 144.3 MB/s  0:00:00
  Attempting uninstall: transformers
    Found existing installation: transformers 5.22.1
    Uninstalling transformers-5.22.1:
      Successfully uninstalled transformers-5.22.1


In [4]:
import time
import pandas as pd
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

In [5]:
# Check that I'm using the right GPU
print(torch.cuda.get_device_name(0))

NVIDIA L4


## 2. Load the model

In [6]:
#model: https://huggingface.co/Qwen/Qwen2.5-7B-Instruct$0
model_name = "Qwen/Qwen2.5-7B-Instruct"

# Structure of this code is based on Assignment 3 & the model's Quickstart guide
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = model = AutoModelForCausalLM.from_pretrained( #parameters from the Quickstart documentation
    model_name,
    torch_dtype="auto",
    device_map="auto"
)

#make sure it loaded on the right device
print("Model loaded on:", model.device)

config.json:   0%|          | 0.00/663 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/27.8k [00:00<?, ?B/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/243 [00:00<?, ?B/s]

Model loaded on: cuda:0


## 3. Helper function: translate one post

Since Qwen is a chat model, we need to send it a system and user message:
- system message saying what its job is
- user message containing the post

In [7]:
# Based on the Quickstart guide here: https://huggingface.co/Qwen/Qwen2.5-7B-Instruct$0
system_prompt = (
    "You are a professional translator. "
    "Translate the user's Chinese social media post into English. "
    "Keep the same tone and emotion as the original. "
    "Output only the English translation, with no notes or explanations."
)


def translate(chinese_text):
    # build the conversation
    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": chinese_text},
    ]

    # Turn into Qwen's expected text format
    prompt_text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True, #from Quickstart guide
    )

    # turn the text into tokens & move to the same device as the model (so it can run quickly)
    inputs = tokenizer(prompt_text, return_tensors="pt").to(model.device)

    # generate the answer
    output_tokens = model.generate(
        **inputs,
        max_new_tokens=256,
        do_sample=False, #always pick the most likely next token so the same post always gets the same translation
    )

    # the output contains the prompt AND the answer, so keep only the new tokens (the answer), then turn them back into text.
    prompt_length = inputs["input_ids"].shape[1]
    answer_tokens = output_tokens[0][prompt_length:]
    english_text = tokenizer.decode(answer_tokens, skip_special_tokens=True) #only decode the answer tokens (not the prompt)

    return english_text.strip()

## 4. Try it on one post

Test out the prompt/messages to make sure it just produced a translation without any notes.

In [8]:
example = "气死姐姐了，快二是阵亡了吗，尼玛，一个半小时过去了也没上车"

start = time.time()
result = translate(example)
seconds = time.time() - start

print("Chinese:", example)
print("English:", result)
print("Took", round(seconds, 1), "seconds")

Chinese: 气死姐姐了，快二是阵亡了吗，尼玛，一个半小时过去了也没上车
English: Auntie is so pissed off, almost two hours have passed and I still haven't gotten on the bus! Damn it!
Took 2.9 seconds


## 5. Run on test posts

In [10]:
import os

save_folder = "/content/drive/MyDrive/ISchool/MIDS/266"

save_every = 50 #I save every 50 posts in case it crashes

# Helper function to translate all of the posts in a file
def translate_matched_file(input_file, output_file):

    # Load the posts
    matched = pd.read_csv(input_file)

    # Translate each post, one at a time
    results = [] # one dictionary per post
    count = 0 # how many posts are done so far

    for index, row in matched.iterrows():
        # Translate Chinese
        english = translate(row["text"])
        count = count + 1

        # Copy every column from the matched file (pair_id, role, label, ...)
        result_row = {}
        for column_name in matched.columns:
            result_row[column_name] = row[column_name]

        # Add our two new columns
        result_row["condition"] = "baseline"
        result_row["translation"] = english

        results.append(result_row)

        # Every 50 posts, save a backup and print progress (save_every = 50, set above)
        if count % save_every == 0:
            pd.DataFrame(results).to_csv(output_file, index=False, encoding="utf-8-sig")
            print("  ", count, "of", len(matched), "done, backup saved")

    # Final save, with all posts
    pd.DataFrame(results).to_csv(output_file, index=False, encoding="utf-8-sig")
    print("  Saved", len(results), "translations to", output_file)


In [11]:
start = time.time()

eval_output = os.path.join(save_folder, "rq1_matched_eval_baseline.csv")
translate_matched_file("rq1_matched_eval.csv", eval_output)

test_output = os.path.join(save_folder, "rq1_matched_test_baseline.csv")
translate_matched_file("rq1_matched_test.csv", test_output)

minutes = (time.time() - start) / 60
print("All done in", round(minutes), "minutes")

   50 of 166 done, backup saved
   100 of 166 done, backup saved
   150 of 166 done, backup saved
  Saved 166 translations to /content/drive/MyDrive/ISchool/MIDS/266/rq1_matched_eval_baseline.csv
   50 of 166 done, backup saved
   100 of 166 done, backup saved
   150 of 166 done, backup saved
  Saved 166 translations to /content/drive/MyDrive/ISchool/MIDS/266/rq1_matched_test_baseline.csv
All done in 15 minutes
